In [3]:
import pandas as pd

dataset = pd.read_csv('it_salaries.csv')
print(f"Số lượng mẫu: {len(dataset)}")
dataset.head(10)

Số lượng mẫu: 100


,SoNamKinhNghiem,DiemTestKyThuat,SoDuAnThucTe,DiemTiengAnh_TOEIC,SoLanNhayViec,Luong_TrieuVND
0,4.8,46.7,5,621,2,38.6
1,11.4,78.7,15,538,3,71.4
2,8.9,61.7,12,796,0,63.2
3,7.4,72.0,9,930,3,58.8
4,2.3,93.1,5,596,2,41.2
5,2.3,58.2,4,425,4,28.4
6,1.2,66.8,1,503,3,27.5
7,10.5,85.0,14,784,4,71.3
8,7.4,57.1,8,435,0,51.6
9,8.6,49.1,12,634,4,54.9


In [5]:
from sklearn.model_selection import train_test_split

# SoNamKinhNghiem,DiemTestKyThuat,SoDuAnThucTe,DiemTiengAnh_TOEIC,SoLanNhayViec,Luong_TrieuVND
featurenames = ['SoNamKinhNghiem', 'DiemTestKyThuat','SoDuAnThucTe','DiemTiengAnh_TOEIC','SoLanNhayViec']

X = dataset[featurenames]
y = dataset['Luong_TrieuVND']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"Số lượng mẫu train: {len(X_train)}, Số lượng mẫu test: {len(X_test)}")

Số lượng mẫu train: 80, Số lượng mẫu test: 20


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

regressor = LinearRegression()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

print(f"Hệ số chặn (w0): {regressor.intercept_}")
i = 1
for name, coef in zip(featurenames, regressor.coef_):
    print(f"Trọng số của {name} (w{i}): {coef}")
    i += 1
    
print("-" * 50)
print(f"R^2 score trên tập Train: {r2_score(y_train, regressor.predict(X_train))}")
print(f"R^2 score trên tập Test: {r2_score(y_test, y_pred)}")
print(f"RMSE trên tập Train: {np.sqrt(mean_squared_error(y_train, regressor.predict(X_train)))}")

Hệ số chặn (w0): 3.4246666659808156
Trọng số của SoNamKinhNghiem (w1): 3.1250098024497914
Trọng số của DiemTestKyThuat (w2): 0.1592464964303446
Trọng số của SoDuAnThucTe (w3): 0.7385750300880476
Trọng số của DiemTiengAnh_TOEIC (w4): 0.01394179962951747
Trọng số của SoLanNhayViec (w5): 0.08719473254383862
--------------------------------------------------
R^2 score trên tập Train: 0.9892506632798358
R^2 score trên tập Test: 0.9781204727412748
RMSE trên tập Train: 1.5076326871063745


In [23]:
def compare(example_idx):
    x = X_test.iloc[example_idx:example_idx + 1]
    y_actual = y_test.iloc[example_idx]
    y_pred = regressor.predict(x)[0]
    diff = abs(y_actual - y_pred)
    print(
        f"Mẫu {example_idx + 1:2d}| "
        f"SoNamKinhNghiem: {x.iloc[0]['SoNamKinhNghiem']} | "
        f"DiemTestKyThuat: {x.iloc[0]['DiemTestKyThuat']} | "
        f"SoDuAnThucTe: {x.iloc[0]['SoDuAnThucTe']} | "
        f"DiemTiengAnh_TOEIC: {x.iloc[0]['DiemTiengAnh_TOEIC']} | "
        f"SoLanNhayViec: {x.iloc[0]['SoLanNhayViec']} | "
        f"Luong_TrieuVND thuc te: {y_actual} | "
        f"Luong_TrieuVND du doan: {y_pred} | "
        f"Chenh lech: {diff} | "
        f"Chenh lech (%): {diff / y_actual * 100} |"
    )

feature_columns = [
    'SoNamKinhNghiem',
    'DiemTestKyThuat',
    'SoDuAnThucTe',
    'DiemTiengAnh_TOEIC',
    'SoLanNhayViec'
]

# Làm bảng so sánh dựa trên số liệu của tập Test
df_compare = pd.DataFrame({
    'SoNamKinhNghiem': [f"{v:,.2f}" for v in X_test['SoNamKinhNghiem']],
    'DiemTestKyThuat': [f"{v:,.2f}" for v in X_test['DiemTestKyThuat']],
    'SoDuAnThucTe': [f"{v:,.2f}" for v in X_test['SoDuAnThucTe']],
    'DiemTiengAnh_TOEIC': [f"{v:,.2f}" for v in X_test['DiemTiengAnh_TOEIC']],
    'SoLanNhayViec': [f"{v:,.2f}" for v in X_test['SoLanNhayViec']],
    'LuongThucTe': [f"{v:,.2f}" for v in y_test],
    'LuongDuDoan': [f"{v:,.2f}" for v in regressor.predict(X_test)]
})

print("\n=== Bảng Đối chiếu Toàn bộ Tập Test (Actual vs Predicted) ===")
print(df_compare.to_string(index=False))


=== Bảng Đối chiếu Toàn bộ Tập Test (Actual vs Predicted) ===
SoNamKinhNghiem DiemTestKyThuat SoDuAnThucTe DiemTiengAnh_TOEIC SoLanNhayViec LuongThucTe LuongDuDoan
           2.80           88.40         3.00             449.00          0.00       32.90       34.73
           8.90           88.30        11.00             368.00          1.00       59.20       58.64
           8.90           61.70        12.00             796.00          0.00       63.20       61.02
          11.10           57.80        15.00             384.00          3.00       66.60       64.01
           8.90           54.20        10.00             539.00          4.00       56.20       55.12
           7.00           78.60         9.00             804.00          4.00       56.00       56.02
           4.00           87.60         5.00             567.00          2.00       42.50       41.65
           9.90           57.00        14.00             897.00          0.00       65.90       66.29
           7.40    

In [12]:
import statsmodels.api as sm

# Thêm cột hệ số tự do w0 (toàn số 1: x0 = 1) vào ma trận X
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_columns

# Huấn luyện mô hình OLS và truyền xname để hiển thị rõ tên từng đặc trưng
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary(xname=all_feature_names))

                            OLS Regression Results                            
Dep. Variable:         Luong_TrieuVND   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.988
Method:                 Least Squares   F-statistic:                     1644.
Date:                Sat, 03 Oct 2026   Prob (F-statistic):           7.93e-90
Time:                        12:28:36   Log-Likelihood:                -183.90
No. Observations:                 100   AIC:                             379.8
Df Residuals:                      94   BIC:                             395.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept (w0)         4.6883      1

In [18]:
# Các đặc trưng có [P]>t (p-value) càng thấp (tiến về) --> đặc trưng quan trọng
# Vì đặc trưng 'SoLanNhayViec' có [P]>t == 0.674 > 0.5 --> đặc trưng này VÔ DỤNG
# Xây dựng lại mô hình sau khi loại bỏ đặc trưng 'SoLanNhayViec'
feature_without_SoLanNhayViec = [col for col in feature_columns if col != 'SoLanNhayViec']
X_without_SoLanNhayViec = dataset[feature_without_SoLanNhayViec]

X_train_wo, X_test_wo, y_train_wo, y_test_wo = train_test_split(
    X_without_SoLanNhayViec,
    y,
    test_size=0.2,
    random_state=1
)

regressor_wo = LinearRegression()
regressor_wo.fit(X_train_wo, y_train_wo) # Huấn luyện dựa trên tập Train
y_pred_wo = regressor_wo.predict(X_test_wo) # Đưa ra dự đoán trên tập Test

print("=== Mô hình sau khi bỏ đặc trưng 'SoLanNhayViec' ===")
print(f'R^2 trên tập Train: {r2_score(y_train_wo, regressor_wo.predict(X_train_wo))}')
print(f'R^2 trên tập Test: {r2_score(y_test_wo, y_pred_wo)}')
print(f'RMSE trên tập Test: {np.sqrt(mean_squared_error(y_test_wo, y_pred_wo))}')

w = [] # Tập hợp các hệ số của phương trình đa biến
w.append(regressor_wo.intercept_)
print(f"Hệ số chặn (w0): {w[0]}")
i = 1
for name, coef in zip(feature_without_SoLanNhayViec, regressor_wo.coef_):
    print(f'Trọng số của {name} (w{i}): {coef}')
    w.append(coef)
    i += 1

# Nếu muốn chạy OLS với tập đặc trưng mới (Không có 'Chiều cao') thì làm như sau:
X_opt_wo = np.append(arr=np.ones((X_without_SoLanNhayViec.shape[0], 1)).astype(float), values=X_without_SoLanNhayViec.values, axis=1)
all_feature_names_wo = ['Intercept (w0)'] + feature_without_SoLanNhayViec
model_ols_wo = sm.OLS(endog=y, exog=X_opt_wo).fit()
print(model_ols_wo.summary(xname=all_feature_names_wo))

=== Mô hình sau khi bỏ đặc trưng 'SoLanNhayViec' ===
R^2 trên tập Train: 0.9872186821595558
R^2 trên tập Test: 0.9937794078768549
RMSE trên tập Test: 1.1440459769753581
Hệ số chặn (w0): 4.876316209462004
Trọng số của SoNamKinhNghiem (w1): 3.12782901991872
Trọng số của DiemTestKyThuat (w2): 0.15660379688042858
Trọng số của SoDuAnThucTe (w3): 0.7439883434009336
Trọng số của DiemTiengAnh_TOEIC (w4): 0.012419001537240307
                            OLS Regression Results                            
Dep. Variable:         Luong_TrieuVND   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.988
Method:                 Least Squares   F-statistic:                     2073.
Date:                Sat, 03 Oct 2026   Prob (F-statistic):           1.79e-91
Time:                        12:37:02   Log-Likelihood:                -184.00
No. Observations:                 100   AIC:                             378.0
Df Residuals:          

In [ ]:
# SoNamKinhNghiem (a1) = 5.0
# DiemTestKyThuat (a2) = 80.0
# SoDuAnThucTe (a3) = 7.0
# DiemTiengAnh_TOEIC (a4): Cần tìm
# MỤC TIÊU: 40.0 <= Luong_TrieuVND <= 50.0  
# Ta cần xác định điểm tiếng anh (TOEIC, IELTS) để có được mức lương mong muốn.

# Hàm quy đổi từ điểm TOEIC ra IELTS
def TOEIC_to_IELTS(TOEIC) -> float:
    if TOEIC < 450:
        return 4.0
    elif TOEIC <= 560:
        return 5.5
    elif TOEIC <= 680:
        return 6.0
    elif TOEIC <= 780:
        return 6.5
    elif TOEIC <= 900:
        return 7.0
    elif TOEIC <= 990:
        return 7.5
    else: 
        return 8.0

a1, a2, a3 = 5.0, 80.0, 7.0    
Luong1, Luong2 = 40.0, 50.0 # 2 trường hợp cho mức lương mong muốn

# Với mức lương 40.0 triệu đồng
DiemTOEIC_1= (Luong1 - w[0] - a1*w[1] - a2*w[2] - a3*w[3])/(w[4])
print(F"Để đạt mức lương 40.0 triệu đồng cần có điểm TOEIC: {DiemTOEIC_1}")
print(F"Để đạt mức lương 40.0 triệu đồng cần có điểm IELTS: {TOEIC_to_IELTS(DiemTOEIC_1)}")

# Với mức lương 50.0 triệu đồng
DiemTOEIC_2= (Luong2 - w[0] - a1*w[1] - a2*w[2] - a3*w[3])/(w[4])
print(F"Để đạt mức lương 50.0 triệu đồng cần có điểm TOEIC: {DiemTOEIC_2}")
print(F"Để đạt mức lương 50.0 triệu đồng cần có điểm IELTS: {TOEIC_to_IELTS(DiemTOEIC_2)}")

Để đạt mức lương 40.0 triệu đồng cần có điểm TOEIC: 140.7775441094015
Để đạt mức lương 40.0 triệu đồng cần có điểm IELTS: 4.0
Để đạt mức lương 50.0 triệu đồng cần có điểm TOEIC: 945.9952558564728
Để đạt mức lương 50.0 triệu đồng cần có điểm IELTS: 7.5
